# GPU-Based Securities | An Underwriting Framework for AI Compute Securitization

## Replication Package for paper "GPU-Based Securities: An Underwriting Framework for AI Compute Securitization" accepted by the Journal of Alternative Investments
**One path edit:** In the first code cell, leave `DATA_DIR = Path(r".")` if the notebook is saved alongside the CSVs, or change it to the single folder containing the original nine unmodified CSV files (without `(1)` in their names). All outputs appear in a `GBS_JAI_outputs` subfolder. Run **Kernel → Restart & Run All**. Python: `numpy`, `pandas`, `matplotlib`, `scipy`, `Pillow`. No proprietary data or network access. Fixed seeds make every exhibit reproducible.

The model describes a **US-law, static, bankruptcy-remote SPV owning installed, separable GPU servers**, with leases to cloud operators. It is a *scenario underwriting tool*, **not** a historical default estimator or rating model. Price quotes are reference points; contract terms, installation/acquisition costs, legal recoverability and refinancing terms are labeled assumed. Monetary figures use a common arbitrary scale: *equipment and installation cost = 100*.

**Audit:** `model_results.csv`, `parameter_audit.csv`, `sources_uses.csv`, `deterministic_cashflows.csv`, sensitivity and investor tables, and all plotted figures are exported at the end. This notebook uses the same common initial random draws across recovery strategies. Initial sponsor contribution of 27 covers 25 of equipment cost plus 2 of prefunded reserve. Debts total 75.

In [1]:
from pathlib import Path
DATA_DIR = Path(r".")  # EDIT ONLY THIS LINE if CSVs are elsewhere, e.g. Path(r"C:\Research\GBS")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.stats import beta
from PIL import Image

OUT = DATA_DIR / "GBS_JAI_outputs"
OUT.mkdir(parents=True, exist_ok=True)
np.set_printoptions(suppress=True, precision=4)
plt.rcParams.update({"figure.dpi":115, "savefig.dpi":220, "font.family":"DejaVu Sans", "font.size":9,
                     "axes.spines.top":False,"axes.spines.right":False,"axes.titleweight":"bold", "axes.labelcolor":"#20324E"})
NAVY="#142A47"; BLUE="#2563A7"; TEAL="#12998E"; GOLD="#D7A32E"; RED="#C65355"; PURPLE="#7454A8"; CYAN="#4FABC9"; GREY="#68798A"
def save(name):
    plt.savefig(OUT / name, bbox_inches="tight", facecolor="white", dpi=220)
    plt.close()
print("Input folder:", DATA_DIR.resolve()); print("Output folder:", OUT.resolve())

Input folder: <DATA_DIR>
Output folder: <DATA_DIR>/GBS_JAI_outputs


## Visual transaction map

**Exhibit 1** is an original programmatic poster-style illustration of the reference SPV and recovery branches. It is generated from matplotlib primitives and is reproduced without image files.

In [2]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
fig,ax=plt.subplots(figsize=(13,6.4));ax.set_xlim(0,13);ax.set_ylim(0,6.4);ax.axis('off');fig.patch.set_facecolor('#F5F8FC');ax.set_facecolor('#F5F8FC')
ax.add_patch(FancyBboxPatch((.2,5.65),12.6,.55,boxstyle='round,pad=0.08,rounding_size=.1',facecolor=NAVY,edgecolor='none'))
ax.text(.50,5.91,'EXHIBIT 1    GPU CASH FLOWS  →  INVESTABLE CREDIT  →  RECOVERY DECISIONS',fontsize=15,color='white',fontweight='bold',va='center')
blocks=[(.30,3.04,2.63,2.05,BLUE,'01  GPU COLLATERAL',['H100 / H200 / B200 fleet','Servers + network installed','US asset-owning SPV']),
        (3.42,3.04,2.63,2.05,TEAL,'02  PAYMENT RIGHTS',['Committed lease minimums','Variable merchant charges','Default and renewal risk']),
        (6.54,3.04,2.63,2.05,PURPLE,'03  PRIORITY WATERFALL',['Fixed / variable operating cost','Senior first; mezzanine next','Reserve + sponsor capital']),
        (9.66,3.04,2.96,2.05,GOLD,'04  EXIT OPTIONS',['Immediate equipment sale','Operating extension','Conditional successor funding'])]
for x,y,w,h,col,label,lines in blocks:
 ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.10,rounding_size=.18',facecolor='white',edgecolor='#DDE6EF',linewidth=1.4))
 ax.add_patch(FancyBboxPatch((x,y+h-.48),w,.48,boxstyle='round,pad=0.01,rounding_size=.14',facecolor=col,edgecolor='none'))
 ax.text(x+.16,y+h-.23,label,color='white',fontsize=10,fontweight='bold',va='center')
 for j,line in enumerate(lines):ax.text(x+.16,y+1.32-j*.38,'• '+line,color=NAVY,fontsize=8.6,va='center')
for x in [3.00,6.12,9.24]:ax.add_patch(FancyArrowPatch((x,4.01),(x+.36,4.01),arrowstyle='-|>',mutation_scale=18,lw=2,color='#66768B'))
ax.add_patch(FancyBboxPatch((.45,.52),12.05,1.61,boxstyle='round,pad=0.18,rounding_size=.13',facecolor='#E6EEF7',edgecolor='none'))
ax.text(.71,1.85,'UNDERWRITING CONDITION: THREE PROTECTIONS MUST WORK TOGETHER',fontsize=11,color=NAVY,fontweight='bold')
for i,(lab,col) in enumerate([('Contract coverage',BLUE),('Operational transferability',TEAL),('Residual value & funding',PURPLE)]):
 x=.71+4*i;ax.add_patch(Circle((x+.15,1.22),.12,facecolor=col,edgecolor='none'));ax.text(x+.41,1.22,lab,fontsize=11,color=NAVY,va='center')
ax.text(.5,.18,'Schematic reference transaction. Arrows show contractual priority, not a claim of an issued GPU ABS.',fontsize=8,color=GREY)
save('exhibit_01_transaction_architecture.png')

## 1. Original data, comparability and transaction evidence

Data are the September 11, 2026 snapshot supplied with the manuscript. The non-spot group is **not** called a long-term contracted rate. Quote-level observations retain provider, billing product and GPU model; executed secondary-market values are kept separate from model-implied B200 values. GPU financing loans and data-center ABS are not relabeled as rated GPU securitizations.

In [3]:
research_map=pd.read_csv(DATA_DIR/'00_research_design_data_map.csv')
specs=pd.read_csv(DATA_DIR/'01_gpu_specs.csv')
rental=pd.read_csv(DATA_DIR/'02_gpu_cloud_rental_prices.csv')
secondary=pd.read_csv(DATA_DIR/'03_gpu_secondary_market.csv')
financing=pd.read_csv(DATA_DIR/'04_gpu_backed_financing_transactions.csv')
operating=pd.read_csv(DATA_DIR/'05_coreweave_operating_panel.csv')
adjacent=pd.read_csv(DATA_DIR/'06_adjacent_datacenter_abs.csv')
registry=pd.read_csv(DATA_DIR/'07_source_registry.csv')
quality=pd.read_csv(DATA_DIR/'08_data_quality_and_design_adjustments.csv')
assert len(rental)==22 and len(specs)==3 and len(financing)==8
assert set(rental['gpu'])=={'H100','H200','B200'}
nonspot=rental.loc[~rental.pricing_type.str.contains('spot|preemptible',case=False,na=False)].copy()
summary=nonspot.groupby('gpu').agg(quotes=('price_per_gpu_hour','size'),providers=('provider','nunique'),median_usd_h=('price_per_gpu_hour','median'),minimum=('price_per_gpu_hour','min'),maximum=('price_per_gpu_hour','max')).reindex(['H100','H200','B200'])
loo=[]
for gpu in ['H100','H200','B200']:
    part=nonspot.loc[nonspot.gpu==gpu]
    for provider in part.provider.unique():
        reduced=part.loc[part.provider!=provider]
        loo.append({'GPU':gpu,'Removed provider':provider,'Remaining quotes':len(reduced),'Median $/GPU-hour':reduced.price_per_gpu_hour.median()})
loo=pd.DataFrame(loo)
summary['leave_one_provider_min']=loo.groupby('GPU')['Median $/GPU-hour'].min()
summary['leave_one_provider_max']=loo.groupby('GPU')['Median $/GPU-hour'].max()
summary.to_csv(OUT/'quotes_summary.csv');loo.to_csv(OUT/'leave_one_provider_out.csv',index=False)
print(summary.round(3).to_string())
print('\nTransaction count:',len(financing),'rows; combined financing rows are not split into invented deals')
print('Ancillary metadata:',len(research_map),'research-map rows,',len(quality),'quality-audit rows')

      quotes  providers  median_usd_h  minimum  maximum  leave_one_provider_min  leave_one_provider_max
gpu                                                                                                    
H100       6          5         4.591     3.49    6.160                    3.99                   5.191
H200       4          4         5.280     4.50    6.305                    4.59                   5.970
B200       6          5         7.875     6.69   12.355                    7.15                   8.600

Transaction count: 8 rows; combined financing rows are not split into invented deals
Ancillary metadata: 7 research-map rows, 7 quality-audit rows


In [4]:
fig,ax=plt.subplots(figsize=(10.6,4.5)); groups=['H100','H200','B200']
provs=list(rental.provider.unique()); cmap=dict(zip(provs,[BLUE,TEAL,GOLD,PURPLE,RED,CYAN]))
for j,g in enumerate(groups):
    part=rental.loc[rental.gpu==g]
    for k,(_,r) in enumerate(part.iterrows()):
        spot=('spot' in str(r.pricing_type).lower() or 'preemptible' in str(r.pricing_type).lower())
        ax.scatter(j+(k-len(part)/2)*.040, r.price_per_gpu_hour, s=62 if not spot else 51,
                   edgecolors=cmap[r.provider],facecolors='white' if spot else cmap[r.provider],linewidths=1.3,zorder=3)
    ax.plot([j-.29,j+.29],[summary.loc[g,'median_usd_h']]*2,color=NAVY,lw=2.5,zorder=4)
    ax.annotate(f"median ${summary.loc[g,'median_usd_h']:.2f}",(j+.31,summary.loc[g,'median_usd_h']),fontsize=8,color=NAVY,va='center')
ax.set_xticks(range(3),groups);ax.set_xlim(-.55,2.60);ax.set_ylim(0,14);ax.set_ylabel('USD per GPU-hour');ax.set_title('Exhibit 3  |  Public rental offers are not identical lease contracts',loc='left',pad=13)
ax.grid(axis='y',alpha=.13)
for p in provs: ax.scatter([],[],color=cmap[p],label=p)
ax.legend(ncol=5,frameon=False,loc='upper center',bbox_to_anchor=(.52,-.14),fontsize=8)
ax.text(.01,-.26,'Filled: non-spot quote; outlined: spot/preemptible. Horizontal bars: non-spot median. Source: original provider CSV.',transform=ax.transAxes,color=GREY,fontsize=8)
save('exhibit_03_rental_quotes.png')

In [5]:
print(secondary[['gpu','variant','observation_type','price_mid_usd','evidence_type']].to_string(index=False))
exec_rows=secondary.loc[secondary.observation_type.str.contains('executed',case=False,na=False)]
modeled_rows=secondary.loc[(secondary.gpu=='B200') & secondary.observation_type.str.contains('model',case=False,na=False)]
H100=float(exec_rows.loc[exec_rows.gpu=='H100','price_mid_usd'].median())
H200=float(exec_rows.loc[exec_rows.gpu=='H200','price_mid_usd'].median())
B200=float(modeled_rows.price_mid_usd.median())
print('Executed H100',H100,'Executed H200',H200,'Model B200',B200)
assert np.isfinite([H100,H200,B200]).all()
fig,ax=plt.subplots(figsize=(10.4,4.25))
sub=secondary.loc[secondary.price_mid_usd.notna()].copy()
colors={'executed_sale_aggregate':TEAL,'broker_listing_aggregate':GOLD,'model_implied':PURPLE}
for obs,part in sub.groupby('evidence_type'):
    ax.scatter(part['gpu'].map({'H100':0,'H200':1,'B200':2})+np.linspace(-.16,.16,len(part)),part.price_mid_usd/1000,s=65,color=colors.get(obs,BLUE),label=obs,alpha=.84)
ax.set_xticks([0,1,2],['H100','H200','B200']);ax.set_ylabel('USD thousands / observation');ax.grid(axis='y',alpha=.14)
ax.set_title('Exhibit 4  |  Secondary-price evidence and evidence type',loc='left',pad=12)
ax.legend(frameon=False,fontsize=8);save('exhibit_04_secondary_evidence.png')

 gpu        variant    observation_type  price_mid_usd            evidence_type
H200 H200 NVL 141GB executed_median_90d          32297  executed_sale_aggregate
H100  H100 NVL 94GB executed_median_90d          24000  executed_sale_aggregate
H100 H100 80GB SXM5 executed_median_90d          16000  executed_sale_aggregate
H100 H100 80GB PCIe executed_median_90d          26682  executed_sale_aggregate
B200 B200 SXM 180GB model_implied_value          86023              model_value
H100       H100 SXM  monthly_avg_resale          32500 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale          28800 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale          26200 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale          24100 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale          22400 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale          20800 broker_listing_aggregate
H100       H100 SXM  monthly_avg_resale 

## 2. Sources and uses, observed-price benchmark, and explicit assumptions

The 50/30/20 mix is **a count-weighted model mix**, not an observed SPV portfolio or asset-value weights. H100/H200 are based on executed secondary-market medians; B200 is a model proxy subjected to a 40% origination valuation haircut. The 30% supporting-equipment uplift is an **assumed** equipment-basis adjustment. The normalized acquisition-plus-deployment cost is an assumed 1.25 times the conservative reference market-equipment basis. These are distinct from post-origination shocks. No input is presented as an actual GPU primary-market invoice. The use of a quote-to-contracted-price realization factor prevents on-demand quotes being misrepresented as enforceable take-or-pay payments.

**Funding identity:** SPV equipment/deployment 100 + reserve 2 = senior 60 + mezzanine 15 + initial equity 27. Initial loan-to-cost: 60% senior, 75% total. Rates represent illustrative *fixed coupon* claims, rather than a claim to match public SOFR margins.

The five price-source firms are CoreWeave, AWS, Lambda, RunPod and Nebius. The $5.45/GPU-hour reference is a count-weighted combination of H100, H200 and B200 non-spot within-generation medians, not a pure direct-neocloud contract price. A 45% realization is about $2.45/GPU-hour and serves as an illustrative low-price underwriting boundary.

In [6]:
MIX={'H100':.50,'H200':.30,'B200':.20}
SECONDARY={'H100':H100,'H200':H200,'B200':B200*.60}
reference_equipment_usd=sum(MIX[g]*SECONDARY[g] for g in MIX)*1.30
hourly=sum(MIX[g]*summary.loc[g,'median_usd_h'] for g in MIX)
REFERENCE_UTIL=.72; COST_MULTIPLE=1.25
public_yield_on_cost=hourly*8760*REFERENCE_UTIL/(reference_equipment_usd*COST_MULTIPLE)
print('Count-weighted rental quote',round(hourly,3), 'USD/GPU-hour')
print('Conservative per-GPU equipment/infrastructure value',round(reference_equipment_usd,0))
print('Indicative gross service revenue / acquisition-deployment cost',f'{public_yield_on_cost:.2%}')
sources=pd.DataFrame({'Uses':['Installed GPU servers / deployment','Prefunded debt-service reserve','Total uses','Senior issuance','Mezzanine issuance','Sponsor paid-in equity','Total sources'], 'Cost units':[100,2,102,60,15,27,102]})
sources.to_csv(OUT/'sources_uses.csv',index=False)
assert sum([60,15,27])==sum([100,2])
PARAMS={
 'Baseline':dict(rent_real=.75,committed=.65,min_floor=.85,contract_month=36,renewal=.35,util_mean=.72,util_sd=.07,price_drift=-.05,price_sd=.14,rho=.45,fixed_annual=.12,variable_cost=.32,depreciation=.22,useful_life=60,tech_prob=.11,tech_price=.84,tech_asset=.76,customer_hazard=.035,anchor=.40,operator_hazard=.010,delay=3,redeploy_rate=.75,transition_cost=.80,sale_fraction=.65,refi_advance=.65,refi_access=.75,refi_floor=.61,refi_fee=.025,legal_failure=.025),
 'Downside':dict(rent_real=.62,committed=.55,min_floor=.80,contract_month=30,renewal=.20,util_mean=.61,util_sd=.10,price_drift=-.10,price_sd=.20,rho=.60,fixed_annual=.15,variable_cost=.38,depreciation=.29,useful_life=54,tech_prob=.20,tech_price=.74,tech_asset=.66,customer_hazard=.075,anchor=.46,operator_hazard=.025,delay=5,redeploy_rate=.62,transition_cost=1.25,sale_fraction=.55,refi_advance=.55,refi_access=.45,refi_floor=.65,refi_fee=.04,legal_failure=.06),
 'Severe':dict(rent_real=.50,committed=.40,min_floor=.75,contract_month=24,renewal=.05,util_mean=.48,util_sd=.13,price_drift=-.16,price_sd=.28,rho=.73,fixed_annual=.18,variable_cost=.42,depreciation=.36,useful_life=48,tech_prob=.30,tech_price=.60,tech_asset=.48,customer_hazard=.14,anchor=.52,operator_hazard=.045,delay=8,redeploy_rate=.43,transition_cost=2.0,sale_fraction=.22,refi_advance=.40,refi_access=.13,refi_floor=.72,refi_fee=.065,legal_failure=.13)}
# A limited commercial squeeze changes ONLY THREE baseline inputs; all remaining assumptions stay unchanged.
moderate=PARAMS['Baseline'].copy()
moderate.update(rent_real=.72, fixed_annual=.125, renewal=.30)
PARAMS={'Baseline':PARAMS['Baseline'],'Moderate':moderate,'Downside':PARAMS['Downside'],'Severe':PARAMS['Severe']}
parameter_rows=[]
for key in PARAMS['Baseline']:
    parameter_rows.append(dict(parameter=key,Baseline=PARAMS['Baseline'][key],Moderate=PARAMS['Moderate'][key],Downside=PARAMS['Downside'][key],Severe=PARAMS['Severe'][key],Evidence='Assumed; stress sensitivity (not estimated default parameters)'))
pd.DataFrame(parameter_rows).to_csv(OUT/'parameter_audit.csv',index=False)
print(sources.to_string(index=False));print('Parameters exported:',len(parameter_rows))

Count-weighted rental quote 5.454 USD/GPU-hour
Conservative per-GPU equipment/infrastructure value 41615.0
Indicative gross service revenue / acquisition-deployment cost 66.13%
                              Uses  Cost units
Installed GPU servers / deployment         100
    Prefunded debt-service reserve           2
                        Total uses         102
                   Senior issuance          60
                Mezzanine issuance          15
            Sponsor paid-in equity          27
                     Total sources         102
Parameters exported: 29


## 3. Vectorized monthly cash-flow waterfall

The stochastic process uses (1) price drift/volatility, (2) utilization with mean reversion, (3) correlated monthly price/utilization shocks, (4) technology events, (5) an anchor lessee's missed payments, and (6) a distinct equipment operator step-in event. These event types are different from an SPV default. Anchor lessee failure interrupts its obligations, creates legal/operational delay and permits recovery **only where title, assignment, hosting and step-in assumptions hold**. Nonrecoverability causes permanent loss of that anchor cash-flow stream. For a simultaneous demand/technology shock, sale and redeployment discounts deteriorate. The model assumes a static pool, no replacement purchases.

**Cash-flow precedence** each month: gross lease receipts → fixed/variable service costs → reserve-funded operational deficits → senior interest/arreas → scheduled senior principal → mezzanine interest/arrears → mezzanine principal if matured or in extension → reserve top-up → equity. The 2-unit reserve is funded by initial paid-in equity. Senior pays monthly amortization over months 1–48, mezzanine has a month-48 bullet. Unpaid amounts are tracked (not erased). Monthly cash flows include any reserve payment. No negative reserve or double-counted sales.

At month 48: `sale` realizes stressed collateral net of costs; `hold` extends unresolved obligations up to month 60 and then sells; `conditional_refi` transfers **all assets and future cash flows** to a successor vehicle for net third-party cash proceeds. Transfer is allowed only if an access/coverage screen passes **and net proceeds fully discharge old debt**. Otherwise the original pool follows the hold extension. No rent beyond transfer date is paid to original investors. The extension produces cash flows through month 60; all strategies are compared on a common 60-month reporting horizon. Investor IRR is calculated from actual payment dates; horizon returns reinvest distributions at 3.5% p.a. Only future *post-contract residual* receipts with at most 12 months of remaining modeled useful life support conditional financing. A conservative third-party advance applied to this net cash-flow PV determines the transfer price, not a free terminal cash gift.

In [7]:
def cash_irr(payments, initial):
    # Same single initial investment and nonnegative later cash flows in every path.
    n=len(payments); m=payments.shape[1]; months=np.arange(1,m+1)
    lo=np.full(n,-.80); hi=np.full(n,.50)
    for _ in range(39):
        mid=(lo+hi)/2
        pv=np.sum(payments/(1+mid[:,None])**months,axis=1)
        higher=pv>initial
        lo=np.where(higher,mid,lo);hi=np.where(higher,hi,mid)
    return np.where(payments.sum(axis=1)>0,(1+(lo+hi)/2)**12-1,-1.0)

def investor_metrics(payments,initial,rf=.035):
    months=np.arange(1,61)
    terminal=(payments*(1+rf/12)**(60-months)).sum(axis=1)
    return (terminal/initial)**(12/60)-1, cash_irr(payments,initial)

def summarize_results(result,scenario,strategy):
    row={'Scenario':scenario,'Exit strategy':strategy,'paths':len(result['senior_loss'])}
    for tr in ['senior','mezz']:
        losses=result[tr+'_loss']
        row[tr+'_EL']=losses.mean()
        row[tr+'_P_loss']=(losses>1e-9).mean()
        row[tr+'_loss_events']=int((losses>1e-9).sum())
        row[tr+'_severity_given_loss']=float(losses[losses>1e-9].mean()) if (losses>1e-9).any() else 0
        row[tr+'_mean_IRR']=result[tr+'_irr'].mean()
        row[tr+'_mean_horizon']=result[tr+'_horizon'].mean()
        row[tr+'_deferred_interest']=(result[tr+'_deferred']>1e-9).mean()
        row[tr+'_any_interest_deferral']=result[tr+'_any_deferral'].mean()
    row['equity_mean_IRR']=result['equity_irr'].mean()
    row['equity_mean_horizon']=result['equity_horizon'].mean()
    row['equity_horizon_p05']=np.quantile(result['equity_horizon'],.05)
    row['economic_life_exit']=result['life_exit'].mean();row['early_operating_exit']=result['early_operating_exit'].mean();row['extension_rate']=result['extended'].mean();row['refi_success_rate']=result['refinanced'].mean()
    return row

In [8]:
def simulate(n=8000,seed=418,params=None,strategy='conditional_refi',record=False):
    p=PARAMS['Baseline'].copy() if params is None else params.copy()
    rng=np.random.default_rng(seed); T=60; maturity=48
    s=np.full(n,60.);m=np.full(n,15.);reserve=np.full(n,2.)
    si=np.zeros(n);mi=np.zeros(n);s_defer=np.zeros(n,bool);m_defer=np.zeros(n,bool)
    s_cf=np.zeros((n,T));m_cf=np.zeros((n,T));e_cf=np.zeros((n,T))
    util=np.full(n,REFERENCE_UTIL);price=np.ones(n);asset=np.full(n,80.)
    tech=np.zeros(n,bool);cust_month=np.full(n,-1,int);op_month=np.full(n,-1,int)
    illegal=np.zeros(n,bool);transition_paid=np.zeros(n,bool)
    sold=np.zeros(n,bool);early_exit=np.zeros(n,bool);life_exit=np.zeros(n,bool);extended=np.zeros(n,bool);refinanced=np.zeros(n,bool)
    joint=np.zeros(n,bool); demand=np.zeros(n,bool);occurrences=np.zeros(n,int)
    monthly_price=(p['price_drift']-p['price_sd']**2/2)/12
    tech_p=1-(1-p['tech_prob'])**(1/12)
    cust_p=1-(1-p['customer_hazard'])**(1/12)
    operator_p=1-(1-p['operator_hazard'])**(1/12)
    debt_initial=75.; base_monthly=100*public_yield_on_cost*p['rent_real']/12
    if record: rows=[]
    for t in range(1,T+1):
        z1=rng.standard_normal(n);z2=rng.standard_normal(n)
        price*=np.exp(monthly_price+p['price_sd']/np.sqrt(12)*z1)
        util=p['util_mean']+.83*(util-p['util_mean'])+p['util_sd']*np.sqrt(1-.83**2)*(p['rho']*z1+np.sqrt(1-p['rho']**2)*z2)
        util=np.clip(util,.16,1.)
        is_low=(util<.55)&(price<.78)
        demand|=is_low
        hit=(~tech)&(rng.random(n)<tech_p)
        tech|=hit;price[hit]*=p['tech_price']
        asset*=(1-p['depreciation'])**(1/12);asset[hit]*=p['tech_asset']
        new_cust=(cust_month<0)&(t<=p['contract_month'])&(rng.random(n)<cust_p*(1+1.9*is_low+1.3*tech.astype(float)))
        cust_month[new_cust]=t
        new_op=(op_month<0)&(rng.random(n)<operator_p*(1+1.5*is_low))
        op_month[new_op]=t
        illegal[new_cust]=rng.random(new_cust.sum())<p['legal_failure']
        joint|=(is_low & tech & (cust_month>0))
        valid=(~sold)&(t<=p['useful_life'])
        contracted=p['committed'] if t<=p['contract_month'] else p['committed']*p['renewal']
        market=1-contracted
        # Contracted minimum is not reduced by low utilization; shortfall in 15% usage flex component can fall.
        lease_pay=base_monthly*contracted*(p['min_floor']+(1-p['min_floor'])*np.clip(util/REFERENCE_UTIL,0,1.25))
        merchant_pay=base_monthly*market*price*np.clip(util/REFERENCE_UTIL,0,1.35)
        live=(cust_month>0)&(t>=cust_month)
        late=live & (t>=cust_month+p['delay']+2*is_low.astype(int)) & ~illegal
        within=live & (~late)
        # Defaulted anchor no longer makes contractual payments; after recovery anchor converts to merchant.
        lease_pay=np.where(live,lease_pay*(1-p['anchor']),lease_pay)
        repl=base_monthly*p['anchor']*p['redeploy_rate']*price*np.clip(util/REFERENCE_UTIL,0,1)
        merchant_pay+=np.where(late,repl,0)
        op_interruption=(op_month>0)&(t-op_month<2)&(t>=op_month)
        revenues=(lease_pay+merchant_pay)*np.where(op_interruption,.55,1)*valid
        fixed=np.where(valid,100*p['fixed_annual']/12,0.)
        operating=fixed+revenues*p['variable_cost']
        # Lower power use does not eliminate hosting and capacity-charge obligations.
        gap=np.maximum(operating-revenues,0)
        funded=np.minimum(reserve,gap);reserve-=funded
        operating_failure=valid&(gap-funded>1e-8)  # unsatisfied fixed bills cause early equipment enforcement
        avail=np.maximum(revenues-operating,0)
        trans=late & ~transition_paid
        fees=np.where(trans,p['transition_cost']*p['anchor'],0.) + np.where(new_op,0.45,0.)
        transition_paid|=trans
        draws=np.minimum(reserve,fees);reserve-=draws
        avail=np.maximum(avail-np.maximum(fees-draws,0),0.)
        due_s=np.where(valid,s*.0625/12+si,si)
        pay=np.minimum(avail,due_s);avail-=pay;due_s-=pay
        take=np.minimum(reserve,due_s);reserve-=take;pay+=take;due_s-=take
        si=due_s;s_defer|=due_s>1e-8;s_cf[:,t-1]+=pay
        schedule=np.minimum(s,60/48) if t<=48 else np.where(extended,s,0.)
        # During an extension, all free cash may repay the remaining senior principal.
        pp=np.minimum(avail,schedule);avail-=pp;s-=pp;s_cf[:,t-1]+=pp
        due_m=np.where(valid,m*.10/12+mi,mi)
        ip=np.minimum(avail,due_m);avail-=ip;due_m-=ip
        mt=np.minimum(reserve,due_m);reserve-=mt;ip+=mt;due_m-=mt
        mi=due_m;m_defer|=due_m>1e-8;m_cf[:,t-1]+=ip
        if t>=48:
            repay=np.minimum(avail,m);avail-=repay;m-=repay;m_cf[:,t-1]+=repay
        # target reserve 2; excess belongs to equity. No promise of a reserve above the prefunded target.
        add=np.minimum(np.maximum(2-reserve,0),avail);reserve+=add;avail-=add
        e_cf[:,t-1]+=avail

        if t<=48:
            # An asset cannot keep earning service rent after the issuer cannot fund hosting.
            # Unfunded hosting costs reduce sale proceeds before secured debt recovers.
            exit_now=operating_failure&(~sold)
            sale_fraction=p['sale_fraction']*np.where(is_low,.78,1)*np.where(tech,.88,1)
            exit_funds=np.where(exit_now,np.maximum(asset*sale_fraction-.6-(gap-funded),0)+reserve,0)
            reserve=np.where(exit_now,0,reserve)
            for debtor,interest,cf in [(s,si,s_cf),(m,mi,m_cf)]:
                pay=np.minimum(exit_funds,interest);interest-=pay;exit_funds-=pay;cf[:,t-1]+=pay
                pay=np.minimum(exit_funds,debtor);debtor-=pay;exit_funds-=pay;cf[:,t-1]+=pay
            e_cf[:,t-1]+=exit_funds
            sold|=exit_now;early_exit|=exit_now
        # Exit at the assumed final month of economic operation (static pool, no replenishment).
        if 48<=t<60 and t>=p['useful_life']:
            end_now=(~sold)
            sale_factor=p['sale_fraction']*np.where(is_low,.78,1)*np.where(tech,.88,1)
            funds=np.where(end_now,np.maximum(asset*sale_factor-.6,0)+reserve,0)
            reserve=np.where(end_now,0,reserve)
            for debtor,interest,cf in [(s,si,s_cf),(m,mi,m_cf)]:
                paid=np.minimum(funds,interest);interest-=paid;funds-=paid;cf[:,t-1]+=paid
                paid=np.minimum(funds,debtor);debtor-=paid;funds-=paid;cf[:,t-1]+=paid
            e_cf[:,t-1]+=funds
            sold|=end_now;life_exit|=end_now
        if record:
            rows.append(dict(month=t,contracted=float(np.mean(lease_pay)),merchant=float(np.mean(merchant_pay)),revenue=float(np.mean(revenues)),cost=float(np.mean(operating)),cash_after_cost=float(np.mean(revenues-operating)),senior_debt=float(np.mean(s)),mezz_debt=float(np.mean(m)),reserve=float(np.mean(reserve)),senior_cash=float(np.mean(s_cf[:,t-1])),mezz_cash=float(np.mean(m_cf[:,t-1])),equity_cash=float(np.mean(e_cf[:,t-1]))))
        if t==48:
            refi_draw=rng.random(n)  # same shock draw in every exit strategy
            sale_factor=p['sale_fraction']*np.where(is_low,.78,1)*np.where(tech,.88,1)
            sale_cash=np.maximum(asset*sale_factor-0.6,0)
            if strategy=='sale':
                closing=sale_cash;take_action=~sold
            elif strategy=='conditional_refi':
                # 12-month remaining-life cap (static pool); based on NET forward cash generation.
                remaining=np.maximum(0,60-t)
                future_net=np.maximum(0,(base_monthly*price*np.clip(util/REFERENCE_UTIL,0,1)*p['redeploy_rate']*(1-p['variable_cost'])-100*p['fixed_annual']/12))
                discounted_months=(1-(1+.10/12)**(-remaining))/(.10/12)
                sale_to_successor=p['refi_advance']*future_net*discounted_months*(1-p['refi_fee'])
                solvent=(sale_to_successor >= s+si+m+mi)
                feasible=(util>p['refi_floor'])&(~illegal)&(refi_draw<p['refi_access']*np.where(is_low,.35,1)*np.where(tech,.60,1))
                take_action=feasible&solvent
                closing=np.where(take_action,sale_to_successor,0.)
                refinanced|=take_action
            else:
                take_action=np.zeros(n,bool);closing=np.zeros(n)
            # Closing funds and reserve repay old creditors, in strict priority.
            closing=np.where(take_action,closing+reserve,0.)
            reserve=np.where(take_action,0,reserve)
            for debtor,interest,cf in [(s,si,s_cf),(m,mi,m_cf)]:
                x=np.minimum(closing,interest);interest-=x;closing-=x;cf[:,t-1]+=x
                x=np.minimum(closing,debtor);debtor-=x;closing-=x;cf[:,t-1]+=x
            e_cf[:,t-1]+=closing
            sold|=take_action
            extended|=(~sold)&((s+si+m+mi)>1e-8)
            if record:
                rows[-1].update(senior_debt=float(np.mean(s)),mezz_debt=float(np.mean(m)),reserve=float(np.mean(reserve)),senior_cash=float(np.mean(s_cf[:,t-1])),mezz_cash=float(np.mean(m_cf[:,t-1])),equity_cash=float(np.mean(e_cf[:,t-1])))
        if t==60:
            # Every untransferred asset is sold after extension. No forward rental PV is credited.
            terminal=np.where(~sold,np.maximum(asset*p['sale_fraction']*np.where(is_low,.78,1)*np.where(tech,.88,1)-.6,0)+reserve,0)
            for debtor,interest,cf in [(s,si,s_cf),(m,mi,m_cf)]:
                x=np.minimum(terminal,interest);interest-=x;terminal-=x;cf[:,t-1]+=x
                x=np.minimum(terminal,debtor);debtor-=x;terminal-=x;cf[:,t-1]+=x
            e_cf[:,t-1]+=terminal
            reserve[:]=0
            if record:
                rows[-1].update(senior_debt=float(np.mean(s)),mezz_debt=float(np.mean(m)),reserve=float(np.mean(reserve)),senior_cash=float(np.mean(s_cf[:,t-1])),mezz_cash=float(np.mean(m_cf[:,t-1])),equity_cash=float(np.mean(e_cf[:,t-1])))
    output={'senior_loss':s/60,'mezz_loss':m/15,'senior_deferred':si/60,'mezz_deferred':mi/15,
            'senior_any_deferral':s_defer,'mezz_any_deferral':m_defer,'extended':extended,'refinanced':refinanced,
            'early_operating_exit':early_exit,'life_exit':life_exit,'low_demand':demand,'joint_distress':joint,'tech_hit':tech,'customer_default':cust_month>0,
            'senior_cf':s_cf,'mezz_cf':m_cf,'equity_cf':e_cf}
    for key,amount in [('senior',60),('mezz',15),('equity',27)]:
        output[key+'_horizon'],output[key+'_irr']=investor_metrics(output[key+'_cf'],amount)
    if record:output['month_table']=pd.DataFrame(rows)
    assert np.all(s>=-1e-7) and np.all(m>=-1e-7) and np.all(reserve>=-1e-7)
    assert np.isfinite(output['senior_horizon']).all()
    return output

## 4. Deterministic cash-flow example with baseline economic drift

For interpretability this path switches off *random* price/utilization innovations, technology jumps, customer defaults and operator interruption, while **retaining the baseline −5% annual rental-price drift and the 35% renewal of committed share after month 36**. It follows the actual conditional-refinancing exit policy; if outside financing does not fully retire the debt, the remaining balance extends beyond month 48 and the collateral closes at month 60. This deterministic path is an explanatory realization, not the simulation mean. Rows 1, 12, 24, 36, 37, 48, 49 and 60 illustrate contract expiry, arrears and terminal distribution.

In [9]:
simple=PARAMS['Baseline'].copy()
for k in ['price_sd','util_sd','tech_prob','customer_hazard','operator_hazard']:
    simple[k]=0
# Retain baseline price drift, contract maturity, renewal, fixed costs and conditional exit terms.
det=simulate(n=1,seed=1234,params=simple,strategy='conditional_refi',record=True)
ex_det=det['month_table'];ex_det.to_csv(OUT/'deterministic_cashflows.csv',index=False)
print(ex_det.loc[ex_det.month.isin([1,12,24,36,37,48,49,60])].round(3).to_string(index=False))
print('Extension:',bool(det['extended'][0]),'Outside refinancing:',bool(det['refinanced'][0]))
print('Terminal principal loss:',det['senior_loss'][0],det['mezz_loss'][0])

 month  contracted  merchant  revenue  cost  cash_after_cost  senior_debt  mezz_debt  reserve  senior_cash  mezz_cash  equity_cash
     1       2.687     1.441    4.127 2.321            1.806        58.75     15.000      2.0        1.562      0.125        0.119
    12       2.687     1.376    4.063 2.300            1.763        45.00     15.000      2.0        1.491      0.125        0.147
    24       2.687     1.309    3.996 2.279            1.717        30.00     15.000      2.0        1.413      0.125        0.179
    36       2.687     1.245    3.932 2.258            1.674        15.00     15.000      2.0        1.335      0.125        0.214
    37       0.940     2.737    3.677 2.177            1.500        13.75     15.000      2.0        1.328      0.125        0.047
    48       0.940     2.614    3.554 2.137            1.417         0.00     14.964      2.0        1.257      0.161        0.000
    49       0.940     2.603    3.544 2.134            1.410         0.00     13.68

## 5. Common-seed Baseline / moderate commercial squeeze / Downside / Severe and three recovery strategies

Scenario differences reflect distinct investment environments and **are not forecasts or estimated scenario probabilities**. For strategy comparisons, all scenarios use the same seed. A path's random sequence still diverges after conditional branching at exit, but until month 48 all simulated operating and market shocks are identical. The event count, Wilson/Jeffreys uncertainty interval and stress sensitivity avoid false precision for very small event rates.

The **Moderate** scenario changes only three commercial inputs: rent realization 75%→72%, annual fixed costs 12%→12.5% of acquisition cost, and committed-share renewal 35%→30%; all other baseline inputs are identical. The Downside and Severe cases jointly shift many variables and are treated as boundary tests, not representative forecasts.

In [10]:
RESULTS={}; rows=[]
for name,p in PARAMS.items():
    for strategy in ['sale','hold','conditional_refi']:
        r=simulate(n=12000,seed=4821,params=p,strategy=strategy)
        RESULTS[(name,strategy)]=r
        rows.append(summarize_results(r,name,strategy))
summary_results=pd.DataFrame(rows)
summary_results.to_csv(OUT/'model_results.csv',index=False)
print(summary_results[['Scenario','Exit strategy','senior_EL','senior_P_loss','mezz_EL','mezz_P_loss','senior_mean_IRR','mezz_mean_IRR','extension_rate','refi_success_rate']].round(4).to_string(index=False))

Scenario    Exit strategy  senior_EL  senior_P_loss  mezz_EL  mezz_P_loss  senior_mean_IRR  mezz_mean_IRR  extension_rate  refi_success_rate
Baseline             sale     0.0012         0.0181   0.1789       0.4922           0.0638         0.0406          0.0000             0.0000
Baseline             hold     0.0009         0.0106   0.0923       0.2061           0.0640         0.0753          0.9981             0.0000
Baseline conditional_refi     0.0009         0.0106   0.0923       0.2061           0.0640         0.0753          0.9861             0.0120
Moderate             sale     0.0049         0.0645   0.3165       0.6152           0.0623        -0.0370          0.0000             0.0000
Moderate             hold     0.0043         0.0464   0.2130       0.3688           0.0628         0.0171          0.9953             0.0000
Moderate conditional_refi     0.0043         0.0464   0.2130       0.3688           0.0628         0.0171          0.9875             0.0078
Downside     

In [11]:
# Conditional losses: conditions are actual simulated states, not mutually exclusive buckets.
condition_rows=[]
for name in PARAMS:
    r=RESULTS[(name,'conditional_refi')]
    for label,mask in [('All paths',np.ones(len(r['senior_loss']),bool)),('Low demand',r['low_demand']),('Technology event',r['tech_hit']),('Lessee default',r['customer_default']),('Joint distress',r['joint_distress'])]:
        if mask.sum()==0:continue
        condition_rows.append({'Scenario':name,'State':label,'N':int(mask.sum()),'Senior P(loss)':(r['senior_loss'][mask]>1e-9).mean(),'Mezz P(loss)':(r['mezz_loss'][mask]>1e-9).mean(),'Senior EL':r['senior_loss'][mask].mean(),'Mezz EL':r['mezz_loss'][mask].mean()})
cond=pd.DataFrame(condition_rows);cond.to_csv(OUT/'conditional_losses.csv',index=False)
print(cond.round(4).to_string(index=False))
# Jeffreys 95% interval for baseline senior events, and event count.
br=RESULTS[('Baseline','conditional_refi')]; k=(br['senior_loss']>1e-9).sum(); n=len(br['senior_loss'])
interval=beta.ppf([.025,.975],k+.5,n-k+.5)
print(f'Baseline senior principal-loss frequency: {k}/{n} = {k/n:.4%}; Jeffreys 95% interval: {interval[0]:.4%}–{interval[1]:.4%}')
# State frequency is the proportion of paths ever entering that state during 60 months.

Scenario            State     N  Senior P(loss)  Mezz P(loss)  Senior EL  Mezz EL
Baseline        All paths 12000          0.0106        0.2061     0.0009   0.0923
Baseline       Low demand  1522          0.0420        0.5066     0.0044   0.2657
Baseline Technology event  5294          0.0213        0.3493     0.0018   0.1689
Baseline   Lessee default  1466          0.0402        0.2428     0.0051   0.1484
Baseline   Joint distress    93          0.1935        0.6989     0.0292   0.5286
Moderate        All paths 12000          0.0464        0.3688     0.0043   0.2130
Moderate       Low demand  1522          0.1583        0.7326     0.0166   0.5021
Moderate Technology event  5294          0.0927        0.5433     0.0086   0.3411
Moderate   Lessee default  1466          0.0975        0.3499     0.0132   0.2451
Moderate   Joint distress    93          0.4301        0.8280     0.0651   0.6891
Downside        All paths 12000          0.9931        0.9989     0.6239   0.9965
Downside       L

In [12]:
fig,axs=plt.subplots(1,3,figsize=(11.5,4.3))
r=RESULTS[('Baseline','conditional_refi')]
for ax,key,label,col in zip(axs,['senior_horizon','mezz_horizon','equity_horizon'],['Senior','Mezzanine','First-loss equity'],[BLUE,TEAL,PURPLE]):
    q=np.quantile(r[key],[.05,.25,.50,.75,.95]);v=np.clip(r[key]*100,-100,150)
    ax.hist(v,bins=38,color=col,alpha=.84,edgecolor='white',lw=.3)
    ax.axvline(q[1]*100,color=NAVY,linestyle='--',lw=1.2)
    ax.axvline(q[2]*100,color=GOLD,linestyle='-',lw=2)
    ax.set_title(label,loc='left');ax.set_xlabel('5-year horizon return (%)');ax.grid(axis='y',alpha=.10)
    ax.text(.02,.94,f"mean {r[key].mean():.1%}\n5th pct {q[0]:.1%}",transform=ax.transAxes,va='top',fontsize=9,color=NAVY,bbox={'facecolor':'white','edgecolor':'none','alpha':.80})
fig.suptitle('Exhibit 9  |  Priority changes the shape of payoff risk',x=.04,y=1.02,ha='left',fontweight='bold',fontsize=12);fig.tight_layout();save('exhibit_09_return_distributions.png')
fig,axs=plt.subplots(1,2,figsize=(10.7,4.3),sharey=True)
for ax,metric,title in zip(axs,['Senior P(loss)','Mezz P(loss)'],['Senior principal impairment','Mezzanine principal impairment']):
    part=cond.loc[cond.Scenario=='Moderate']; ax.barh(np.arange(len(part)),part[metric].values*100,color=[BLUE,TEAL,GOLD,RED,PURPLE]);ax.invert_yaxis()
    ax.set_yticks(np.arange(len(part)),part.State);ax.set_xlabel('Probability of principal loss (%)');ax.set_title(title,loc='left');ax.grid(axis='x',alpha=.10)
fig.suptitle('Exhibit 10  |  Conditional loss in moderate stress states',x=.05,y=1.03,ha='left',fontsize=12,fontweight='bold');fig.tight_layout();save('exhibit_10_conditional_losses.png')

In [13]:
rec=summary_results.loc[summary_results.Scenario=='Baseline'].copy()
rec.to_csv(OUT/'recovery_comparison.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(10.6,4.1))
for ax,m,title in zip(axs,['senior_EL','mezz_EL'],['Senior expected principal loss','Mezzanine expected principal loss']):
    ax.bar(['Immediate sale','Operate + extension','Conditional refinancing'],rec[m].values*100,color=[RED,GOLD,TEAL])
    ax.set_title(title,loc='left');ax.set_ylabel('Loss (% of original principal)');ax.tick_params(axis='x',rotation=10);ax.grid(axis='y',alpha=.13)
fig.suptitle('Exhibit 11  |  Recovery options compared under identical baseline shocks',x=.05,y=1.04,ha='left',fontsize=12,fontweight='bold');fig.tight_layout();save('exhibit_11_recovery_comparison.png')

## 6. Sensitivity analysis and financeability boundary

To preserve the fixed reference capital structure, the grid varies *achieved contracted-price realization* and *fixed annual cost burden* without mechanically reducing original debt when collateral is impaired. A second table varies origination debt size and converts returns only from the risk-adjusted cash-flow model, not market valuation. Recovery break even is evaluated against a separate standalone 12-month redeployment cash-flow value, net of transition expense; **it is not double counted in the lender waterfall**.

In [14]:
grid_rows=[]
for rr in [.45,.55,.65,.75,.85]:
    for fc in [.08,.12,.16,.20]:
        p=PARAMS['Baseline'].copy();p['rent_real']=rr;p['fixed_annual']=fc
        r=simulate(n=1800,seed=860,params=p,strategy='conditional_refi')
        grid_rows.append({'Realization':rr,'Fixed cost / original cost':fc,'Senior EL':r['senior_loss'].mean(),'Mezz EL':r['mezz_loss'].mean(),'Senior P(loss)':(r['senior_loss']>1e-9).mean(),'Mezz P(loss)':(r['mezz_loss']>1e-9).mean()})
grid=pd.DataFrame(grid_rows);grid.to_csv(OUT/'financeability_grid.csv',index=False)
pv=grid.pivot(index='Fixed cost / original cost',columns='Realization',values='Mezz EL')
fig,ax=plt.subplots(figsize=(9.4,4.6))
im=ax.imshow(pv.values*100,origin='lower',aspect='auto',cmap='Spectral_r',vmin=0,vmax=max(15,pv.values.max()*100))
for i in range(len(pv.index)):
 for j in range(len(pv.columns)):
    value=pv.iloc[i,j]*100
    ax.text(j,i,f'{value:.1f}%',ha='center',va='center',color='white' if value>18 else NAVY,fontsize=9,fontweight='bold')
ax.set_xticks(range(len(pv.columns)),[f'{x:.0%}' for x in pv.columns]);ax.set_yticks(range(len(pv.index)),[f'{x:.0%}' for x in pv.index]);ax.set_xlabel('Actual contract rate / observed public quote');ax.set_ylabel('Fixed annual cost / initial investment');ax.set_title('Exhibit 12  |  Financeability map: mezzanine expected principal loss',loc='left',pad=14)
fig.colorbar(im,ax=ax,label='Mezzanine expected principal loss (%)');save('exhibit_12_financeability_heatmap.png')
# Illustrative break-even, independent of sale-vs-refinance transfer mechanics; useful-life limited to 12 months.
be=[]
for lease in np.linspace(.30,1.0,15):
 for delay in [1,3,6,9]:
    net_m=100*public_yield_on_cost*lease/12*(1-PARAMS['Baseline']['variable_cost'])-100*PARAMS['Baseline']['fixed_annual']/12
    op_value=max(0,net_m)*max(0,12-delay)*((1+.10/12)**(-(12-delay)/2))-.8
    # Ownership retained: include end-of-holding-period resale value, discounted to month 48.
    residual_at_60=80*(1-.22)**5*.65-.6
    op_value+=max(0,residual_at_60)/(1+.10/12)**12
    sale_value=80*(1-.22)**4*.65-.6
    be.append({'Realization':lease,'Delay months':delay,'PV 12m net operation':op_value,'Sale proceeds at 48m':sale_value,'Redeployment minus sale':op_value-sale_value})
break_even=pd.DataFrame(be);break_even.to_csv(OUT/'redeployment_break_even.csv',index=False)
fig,ax=plt.subplots(figsize=(9.7,4.3))
for delay,col in [(1,TEAL),(3,BLUE),(6,GOLD),(9,RED)]:
 part=break_even.loc[break_even['Delay months']==delay]
 ax.plot(part.Realization*100,part['Redeployment minus sale'],label=f'{delay}-month delay',color=col,lw=2.4)
ax.axhline(0,color=NAVY,lw=1,linestyle='--');ax.set_xlabel('Realized rental rate / public quote (%)');ax.set_ylabel('12-month redeployment NPV – sale proceeds (cost units)')
ax.set_title('Exhibit 13  |  When redeployment adds value relative to immediate sale',loc='left',pad=13)
ax.legend(ncol=4,frameon=False,fontsize=8);ax.grid(alpha=.12);save('exhibit_13_redeployment_breakeven.png')

## 7. One-input-at-a-time investor stress attribution

Each case changes one parameter from the Baseline while all others—including the 60/15 initial debt amounts—remain unchanged. Identical random seeds and 5,000 paths isolate economic sensitivity from multi-parameter stress bundles. This also provides a direct link between a lender's diligence findings and modeled note impairment. The effects are *not* additive because the waterfall has nonlinear protections.

In [15]:
# One-way adverse changes around the Baseline: fixed debt size, matched random seed.
one_way_specs=[
    ('Lower realized rent: 75% to 68%','rent_real',.68),
    ('Higher fixed cost: 12% to 14.5%','fixed_annual',.145),
    ('Lower contract renewal: 35% to 20%','renewal',.20),
    ('Lower mean utilization: 72% to 64%','util_mean',.64),
    ('Higher lessee default hazard: 3.5% to 7.5%','customer_hazard',.075),
    ('Lower liquidation realization: 65% to 52%','sale_fraction',.52),
]
one_way_rows=[]
b0=simulate(n=5000,seed=4821,params=PARAMS['Baseline'],strategy='conditional_refi')
one_way_rows.append({'Test':'Baseline','Changed parameter':'None','Senior EL':b0['senior_loss'].mean(),'Mezz EL':b0['mezz_loss'].mean(),'Mezz P(loss)':(b0['mezz_loss']>1e-9).mean()})
for label,key,value in one_way_specs:
    pp=PARAMS['Baseline'].copy();pp[key]=value
    rr=simulate(n=5000,seed=4821,params=pp,strategy='conditional_refi')
    one_way_rows.append({'Test':label,'Changed parameter':key,'Senior EL':rr['senior_loss'].mean(),'Mezz EL':rr['mezz_loss'].mean(),'Mezz P(loss)':(rr['mezz_loss']>1e-9).mean()})
one_way=pd.DataFrame(one_way_rows)
one_way.to_csv(OUT/'one_way_sensitivity.csv',index=False)
print(one_way.round(4).to_string(index=False))
base_mezz=one_way.loc[0,'Mezz EL']
sub=one_way.iloc[1:].copy().sort_values('Mezz EL')
fig,ax=plt.subplots(figsize=(10.6,4.3))
y=np.arange(len(sub))
ax.barh(y,sub['Mezz EL'].values*100,color=[TEAL,BLUE,PURPLE,GOLD,RED,CYAN],height=.65)
ax.axvline(base_mezz*100,color=NAVY,lw=1.7,ls='--',label=f'Baseline: {base_mezz:.1%}')
ax.set_yticks(y,sub.Test);ax.set_xlabel('Expected mezzanine principal loss (% of initial principal)')
ax.set_title('Exhibit 20  |  Isolated underwriting pressures, one input changed at a time',loc='left',pad=12)
ax.set_xlim(0,max(30,sub['Mezz EL'].max()*110));ax.grid(axis='x',alpha=.12)
ax.legend(loc='lower right',frameon=False)
for yi,v in enumerate(sub['Mezz EL'].values*100):ax.text(v+.7,yi,f'{v:.1f}%',va='center',color=NAVY,fontsize=8)
fig.tight_layout();save('exhibit_20_one_way_sensitivity.png')

                                      Test Changed parameter  Senior EL  Mezz EL  Mezz P(loss)
                                  Baseline              None     0.0010   0.0943        0.2158
           Lower realized rent: 75% to 68%         rent_real     0.0077   0.3082        0.4862
           Higher fixed cost: 12% to 14.5%      fixed_annual     0.0085   0.2913        0.4560
        Lower contract renewal: 35% to 20%           renewal     0.0017   0.1356        0.2712
        Lower mean utilization: 72% to 64%         util_mean     0.0031   0.2048        0.3826
Higher lessee default hazard: 3.5% to 7.5%   customer_hazard     0.0012   0.0973        0.2144
 Lower liquidation realization: 65% to 52%     sale_fraction     0.0015   0.1268        0.2698


## 7. Additional accounting and consistency checks

Reported statistics are computed from precisely the same result objects used in figures; file outputs retain full unrounded precision. Event counts and an interval are shown for rare events. The interest arrears variables distinguish current-payment deferral from ultimate principal loss. The model records a *US-law conditional legal/operational structure* as a modeling premise; evidence of actual lender entitlements would require specific transaction documents.

Audit flags below check source/use equality, no negative principal, no early sale after transfer, and that reports are created. `run_log.txt` records the exact parameters and software versions. Any sensitivity parameter not in the original CSV is intentionally marked as `Assumed` in `parameter_audit.csv`.

In [16]:
# Investor result snapshots and formal audit.
deferrals=summary_results.loc[summary_results['Exit strategy']=='conditional_refi', ['Scenario','senior_any_interest_deferral','mezz_any_interest_deferral','senior_deferred_interest','mezz_deferred_interest','early_operating_exit','economic_life_exit','extension_rate']]
deferrals.to_csv(OUT/'interest_and_resolution_states.csv',index=False)
final=summary_results.loc[summary_results['Exit strategy']=='conditional_refi'].copy()
final.to_csv(OUT/'investor_summary.csv',index=False)
assert abs(sources.loc[sources.Uses=='Total sources','Cost units'].iloc[0]-sources.loc[sources.Uses=='Total uses','Cost units'].iloc[0])<1e-10
assert all(np.isfinite(final.select_dtypes('number').to_numpy().ravel()))
assert len(grid)==20 and len(cond)>0
report=['Source/use balance: PASS','All expected shortfalls finite: PASS',f'Price quotes: {len(rental)} (nonspot {len(nonspot)})',f'Public GPU financing rows: {len(financing)}',f'Observed secondary records: {len(secondary)}',f'Public quote-equivalent gross revenue/cost: {public_yield_on_cost:.5f}',f'Baseline senior principal loss events: {k}/{n}',f'Jeffreys 95% lower/upper: {interval[0]:.7f}, {interval[1]:.7f}',f'Numpy: {np.__version__}',f'Pandas: {pd.__version__}']
(OUT/'run_log.txt').write_text('\n'.join(report),encoding='utf-8')
for line in report:print(line)
print('\nCompleted: figures and CSV outputs are in',OUT)

Source/use balance: PASS
All expected shortfalls finite: PASS
Price quotes: 22 (nonspot 16)
Public GPU financing rows: 8
Observed secondary records: 12
Public quote-equivalent gross revenue/cost: 0.66131
Baseline senior principal loss events: 127/12000
Jeffreys 95% lower/upper: 0.0088684, 0.0125343
Numpy: 2.3.5
Pandas: 2.2.3

Completed: figures and CSV outputs are in <DATA_DIR>/GBS_JAI_outputs
